In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Sample Data (replace with your actual dataset)
data = {
    'YearsExperience': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'EducationLevel': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'High School', 'High School', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master'],
    'JobTitle': ['Developer', 'Developer', 'Lead', 'Manager', 'Developer', 'Lead', 'Developer', 'Manager', 'Lead', 'Developer', 'Support', 'Developer', 'Manager', 'Lead', 'Developer', 'Support', 'Developer', 'Manager', 'Lead', 'Developer'],
    'Location': ['Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2'],
    'Salary': [50000, 60000, 80000, 100000, 70000, 90000, 55000, 110000, 95000, 75000, 40000, 50000, 80000, 100000, 75000, 45000, 58000, 115000, 98000, 78000]
}
df = pd.DataFrame(data)

# Define features (X) and target (y)
X = df.drop('Salary', axis=1)
y = df['Salary']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Create the full pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model_pipeline.fit(X_train, y_train)

# Evaluate the model (optional, but good practice)
train_score = model_pipeline.score(X_train, y_train)
test_score = model_pipeline.score(X_test, y_test)
print(f"Train R2 Score: {train_score:.2f}")
print(f"Test R2 Score: {test_score:.2f}")

# Save the trained model
joblib.dump(model_pipeline, 'salary_prediction_model.pkl')
print("Model trained and saved as salary_prediction_model.pkl")

Train R2 Score: 0.95
Test R2 Score: 0.39
Model trained and saved as salary_prediction_model.pkl


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import time

print("--- Enhanced Model Training Script ---")

# ==============================================================================
# 1. GENERATE A LARGER, MORE REALISTIC SYNTHETIC DATASET
# ==============================================================================
def generate_synthetic_data(n_samples=1000):
    """Creates a rich dataset with more features and realistic salary logic."""
    print(f"Generating {n_samples} synthetic data samples...")
    
    np.random.seed(42) # for reproducibility

    education_levels = ['High School', 'Bachelor', 'Master', 'PhD']
    job_titles = ['Support', 'Developer', 'Lead', 'Manager']
    locations = ['Tier3', 'Tier2', 'Tier1']
    industries = ['Retail', 'Finance', 'Tech', 'Healthcare']
    company_sizes = ['Small (<50)', 'Medium (50-1000)', 'Large (>1000)']

    data = {
        'YearsExperience': np.random.uniform(0.5, 25, n_samples).round(1),
        'EducationLevel': np.random.choice(education_levels, n_samples, p=[0.1, 0.4, 0.35, 0.15]),
        'JobTitle': np.random.choice(job_titles, n_samples, p=[0.15, 0.45, 0.25, 0.15]),
        'Location': np.random.choice(locations, n_samples, p=[0.2, 0.4, 0.4]),
        'Industry': np.random.choice(industries, n_samples, p=[0.15, 0.25, 0.4, 0.2]),
        'CompanySize': np.random.choice(company_sizes, n_samples, p=[0.3, 0.5, 0.2])
    }
    df = pd.DataFrame(data)

    base_salary = 300000
    education_premiums = {'High School': 1, 'Bachelor': 1.2, 'Master': 1.5, 'PhD': 1.8}
    job_premiums = {'Support': 1, 'Developer': 1.3, 'Lead': 1.7, 'Manager': 2.2}
    location_premiums = {'Tier3': 0.85, 'Tier2': 1, 'Tier1': 1.25}
    industry_premiums = {'Retail': 0.9, 'Healthcare': 1.0, 'Finance': 1.2, 'Tech': 1.3}
    company_size_premiums = {'Small (<50)': 0.9, 'Medium (50-1000)': 1, 'Large (>1000)': 1.15}

    df['Salary'] = (base_salary + (df['YearsExperience'] * 25000)) * \
                   df['EducationLevel'].map(education_premiums) * \
                   df['JobTitle'].map(job_premiums) * \
                   df['Location'].map(location_premiums) * \
                   df['Industry'].map(industry_premiums) * \
                   df['CompanySize'].map(company_size_premiums)

    noise = np.random.normal(0, 0.08, n_samples)
    df['Salary'] *= (1 + noise)
    df['Salary'] = df['Salary'].astype(int)

    print("Synthetic data generated successfully.")
    return df

df = generate_synthetic_data(n_samples=1000)
X = df.drop('Salary', axis=1)
y = df['Salary']

# ==============================================================================
# 2. DEFINE THE PREPROCESSING AND MODEL PIPELINE
# ==============================================================================
numerical_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================================================================
# 3. HYPERPARAMETER TUNING WITH GridSearchCV
# ==============================================================================
print("\nStarting hyperparameter tuning with GridSearchCV...")
start_time = time.time()

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

end_time = time.time()
print(f"Hyperparameter tuning finished in {end_time - start_time:.2f} seconds.")

best_model = grid_search.best_estimator_
print("\nBest parameters found:", grid_search.best_params_)

# ==============================================================================
# 4. EVALUATE THE FINAL, TUNED MODEL
# ==============================================================================
print("\nEvaluating the best model on the test set...")
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Model Performance ---")
print(f"R-squared (R²): {r2:.3f}")
print(f"Mean Absolute Error (MAE): ₹{mae:,.0f}")
print(f"-> The model explains {r2:.1%} of the variance in salary.")
print(f"-> On average, the model's prediction is off by approximately ₹{mae:,.0f}.")

# ==============================================================================
# 5. SAVE THE ENHANCED MODEL
# ==============================================================================
model_filename = 'enhanced_salary_model.pkl'
joblib.dump(best_model, model_filename)
print(f"\n✅ Enhanced model successfully trained and saved as '{model_filename}'")

# =============================
# 6. BASELINE MODEL FOR COMPARISON
# =============================
print("\n--- Training Baseline Model for Comparison ---")

# Use only YearsExperience and EducationLevel as features
X_base = df[['YearsExperience', 'EducationLevel']]

# Preprocessing for baseline: scale YearsExperience, one-hot encode EducationLevel
base_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['YearsExperience']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['EducationLevel'])
])

base_pipeline = Pipeline([
    ('preprocessor', base_preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

base_pipeline.fit(X_base_train, y_base_train)

# Evaluate baseline model
base_y_pred = base_pipeline.predict(X_base_test)
base_r2 = r2_score(y_base_test, base_y_pred)
base_mae = mean_absolute_error(y_base_test, base_y_pred)

print("\n--- Baseline Model Performance ---")
print(f"R-squared (R²): {base_r2:.3f}")
print(f"Mean Absolute Error (MAE): ₹{base_mae:,.0f}")

# Save baseline model
base_model_filename = 'baseline_salary_model.pkl'
joblib.dump(base_pipeline, base_model_filename)
print(f"\n✅ Baseline model successfully trained and saved as '{base_model_filename}'")

--- Enhanced Model Training Script ---
Generating 1000 synthetic data samples...
Synthetic data generated successfully.

Starting hyperparameter tuning with GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Hyperparameter tuning finished in 31.48 seconds.

Best parameters found: {'regressor__max_depth': 20, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}

Evaluating the best model on the test set...

--- Model Performance ---
R-squared (R²): 0.860
Mean Absolute Error (MAE): ₹196,720
-> The model explains 86.0% of the variance in salary.
-> On average, the model's prediction is off by approximately ₹196,720.

✅ Enhanced model successfully trained and saved as 'enhanced_salary_model.pkl'

--- Training Baseline Model for Comparison ---

--- Baseline Model Performance ---
R-squared (R²): 0.293
Mean Absolute Error (MAE): ₹498,899

✅ Baseline model successfully trained and saved as 'baseline_salary_model.pkl'


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import time

print("--- Enhanced Model Training Script ---")

# ==============================================================================
# 1. GENERATE A LARGER, MORE REALISTIC SYNTHETIC DATASET
# ==============================================================================
def generate_synthetic_data(n_samples=1000):
    """Creates a rich dataset with more features and realistic salary logic."""
    print(f"Generating {n_samples} synthetic data samples...")
    
    np.random.seed(42) # for reproducibility

    education_levels = ['High School', 'Bachelor', 'Master', 'PhD']
    job_titles = ['Support', 'Developer', 'Lead', 'Manager']
    locations = ['Tier3', 'Tier2', 'Tier1']
    industries = ['Retail', 'Finance', 'Tech', 'Healthcare']
    company_sizes = ['Small (<50)', 'Medium (50-1000)', 'Large (>1000)']

    data = {
        'YearsExperience': np.random.uniform(0.5, 25, n_samples).round(1),
        'EducationLevel': np.random.choice(education_levels, n_samples, p=[0.1, 0.4, 0.35, 0.15]),
        'JobTitle': np.random.choice(job_titles, n_samples, p=[0.15, 0.45, 0.25, 0.15]),
        'Location': np.random.choice(locations, n_samples, p=[0.2, 0.4, 0.4]),
        'Industry': np.random.choice(industries, n_samples, p=[0.15, 0.25, 0.4, 0.2]),
        'CompanySize': np.random.choice(company_sizes, n_samples, p=[0.3, 0.5, 0.2])
    }
    df = pd.DataFrame(data)

    base_salary = 300000
    education_premiums = {'High School': 1, 'Bachelor': 1.2, 'Master': 1.5, 'PhD': 1.8}
    job_premiums = {'Support': 1, 'Developer': 1.3, 'Lead': 1.7, 'Manager': 2.2}
    location_premiums = {'Tier3': 0.85, 'Tier2': 1, 'Tier1': 1.25}
    industry_premiums = {'Retail': 0.9, 'Healthcare': 1.0, 'Finance': 1.2, 'Tech': 1.3}
    company_size_premiums = {'Small (<50)': 0.9, 'Medium (50-1000)': 1, 'Large (>1000)': 1.15}

    df['Salary'] = (base_salary + (df['YearsExperience'] * 25000)) * \
                   df['EducationLevel'].map(education_premiums) * \
                   df['JobTitle'].map(job_premiums) * \
                   df['Location'].map(location_premiums) * \
                   df['Industry'].map(industry_premiums) * \
                   df['CompanySize'].map(company_size_premiums)

    noise = np.random.normal(0, 0.08, n_samples)
    df['Salary'] *= (1 + noise)
    df['Salary'] = df['Salary'].astype(int)

    print("Synthetic data generated successfully.")
    return df

df = generate_synthetic_data(n_samples=1000)
X = df.drop('Salary', axis=1)
y = df['Salary']

# ==============================================================================
# 2. DEFINE THE PREPROCESSING AND MODEL PIPELINE
# ==============================================================================
numerical_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================================================================
# 3. HYPERPARAMETER TUNING WITH GridSearchCV
# ==============================================================================
print("\nStarting hyperparameter tuning with GridSearchCV...")
start_time = time.time()

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

end_time = time.time()
print(f"Hyperparameter tuning finished in {end_time - start_time:.2f} seconds.")

best_model = grid_search.best_estimator_
print("\nBest parameters found:", grid_search.best_params_)

# ==============================================================================
# 4. EVALUATE THE FINAL, TUNED MODEL
# ==============================================================================
print("\nEvaluating the best model on the test set...")
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Model Performance ---")
print(f"R-squared (R²): {r2:.3f}")
print(f"Mean Absolute Error (MAE): ₹{mae:,.0f}")
print(f"-> The model explains {r2:.1%} of the variance in salary.")
print(f"-> On average, the model's prediction is off by approximately ₹{mae:,.0f}.")

# ==============================================================================
# 5. SAVE THE ENHANCED MODEL
# ==============================================================================
model_filename = 'enhanced_salary_model.pkl'
joblib.dump(best_model, model_filename)
print(f"\n✅ Enhanced model successfully trained and saved as '{model_filename}'")

# =============================
# 6. BASELINE MODEL FOR COMPARISON
# =============================
print("\n--- Training Baseline Model for Comparison ---")

# Use only YearsExperience and EducationLevel as features
X_base = df[['YearsExperience', 'EducationLevel']]

# Preprocessing for baseline: scale YearsExperience, one-hot encode EducationLevel
base_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['YearsExperience']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['EducationLevel'])
])

base_pipeline = Pipeline([
    ('preprocessor', base_preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

base_pipeline.fit(X_base_train, y_base_train)

# Evaluate baseline model
base_y_pred = base_pipeline.predict(X_base_test)
base_r2 = r2_score(y_base_test, base_y_pred)
base_mae = mean_absolute_error(y_base_test, base_y_pred)

print("\n--- Baseline Model Performance ---")
print(f"R-squared (R²): {base_r2:.3f}")
print(f"Mean Absolute Error (MAE): ₹{base_mae:,.0f}")

# Save baseline model
base_model_filename = 'baseline_salary_model.pkl'
joblib.dump(base_pipeline, base_model_filename)
print(f"\n✅ Baseline model successfully trained and saved as '{base_model_filename}'")

--- Enhanced Model Training Script ---
Generating 1000 synthetic data samples...
Synthetic data generated successfully.

Starting hyperparameter tuning with GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Hyperparameter tuning finished in 34.14 seconds.

Best parameters found: {'regressor__max_depth': 20, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}

Evaluating the best model on the test set...

--- Model Performance ---
R-squared (R²): 0.860
Mean Absolute Error (MAE): ₹196,720
-> The model explains 86.0% of the variance in salary.
-> On average, the model's prediction is off by approximately ₹196,720.

✅ Enhanced model successfully trained and saved as 'enhanced_salary_model.pkl'

--- Training Baseline Model for Comparison ---

--- Baseline Model Performance ---
R-squared (R²): 0.293
Mean Absolute Error (MAE): ₹498,899

✅ Baseline model successfully trained and saved as 'baseline_salary_model.pkl'


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

print("Starting model training process...")

# Sample Data (In a real project, you would load this from a CSV file)
data = {
    'YearsExperience': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'EducationLevel': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'High School', 'High School', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master'],
    'JobTitle': ['Developer', 'Developer', 'Lead', 'Manager', 'Developer', 'Lead', 'Developer', 'Manager', 'Lead', 'Developer', 'Support', 'Developer', 'Manager', 'Lead', 'Developer', 'Support', 'Developer', 'Manager', 'Lead', 'Developer'],
    'Location': ['Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2', 'Tier1', 'Tier3', 'Tier1', 'Tier2'],
    'Salary': [50000, 60000, 80000, 100000, 70000, 90000, 55000, 110000, 95000, 75000, 40000, 50000, 80000, 100000, 75000, 45000, 58000, 115000, 98000, 78000]
}
df = pd.DataFrame(data)

# Define features (X) and target (y)
X = df.drop('Salary', axis=1)
y = df['Salary']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Create preprocessing pipelines for numerical and categorical features
# StandardScaler for numerical data, OneHotEncoder for categorical data
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Create a preprocessor to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough' # Keep other columns (if any)
)

# Create the full machine learning pipeline
# This pipeline will first preprocess the data, then feed it to the regressor
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
print("Training the model...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")

# Evaluate the model (optional, but good practice)
score = model_pipeline.score(X_test, y_test)
print(f"Model R^2 Score on Test Data: {score:.2f}")

# Save the trained pipeline to a file
joblib.dump(model_pipeline, 'salary_prediction_model.pkl')
print("Model pipeline saved successfully as salary_prediction_model.pkl")

Starting model training process...
Training the model...
Model training complete.
Model R^2 Score on Test Data: 0.39
Model pipeline saved successfully as salary_prediction_model.pkl


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import time

print("--- Enhanced Model Training Script ---")

# ==============================================================================
# 1. GENERATE A LARGER, MORE REALISTIC SYNTHETIC DATASET
# ==============================================================================
def generate_synthetic_data(n_samples=1000):
    """Creates a rich dataset with more features and realistic salary logic."""
    print(f"Generating {n_samples} synthetic data samples...")
    
    np.random.seed(42) # for reproducibility

    education_levels = ['High School', 'Bachelor', 'Master', 'PhD']
    job_titles = ['Support', 'Developer', 'Lead', 'Manager']
    locations = ['Tier3', 'Tier2', 'Tier1']
    industries = ['Retail', 'Finance', 'Tech', 'Healthcare']
    company_sizes = ['Small (<50)', 'Medium (50-1000)', 'Large (>1000)']

    data = {
        'YearsExperience': np.random.uniform(0.5, 25, n_samples).round(1),
        'EducationLevel': np.random.choice(education_levels, n_samples, p=[0.1, 0.4, 0.35, 0.15]),
        'JobTitle': np.random.choice(job_titles, n_samples, p=[0.15, 0.45, 0.25, 0.15]),
        'Location': np.random.choice(locations, n_samples, p=[0.2, 0.4, 0.4]),
        'Industry': np.random.choice(industries, n_samples, p=[0.15, 0.25, 0.4, 0.2]),
        'CompanySize': np.random.choice(company_sizes, n_samples, p=[0.3, 0.5, 0.2])
    }
    df = pd.DataFrame(data)

    base_salary = 300000
    education_premiums = {'High School': 1, 'Bachelor': 1.2, 'Master': 1.5, 'PhD': 1.8}
    job_premiums = {'Support': 1, 'Developer': 1.3, 'Lead': 1.7, 'Manager': 2.2}
    location_premiums = {'Tier3': 0.85, 'Tier2': 1, 'Tier1': 1.25}
    industry_premiums = {'Retail': 0.9, 'Healthcare': 1.0, 'Finance': 1.2, 'Tech': 1.3}
    company_size_premiums = {'Small (<50)': 0.9, 'Medium (50-1000)': 1, 'Large (>1000)': 1.15}

    df['Salary'] = (base_salary + (df['YearsExperience'] * 25000)) * \
                   df['EducationLevel'].map(education_premiums) * \
                   df['JobTitle'].map(job_premiums) * \
                   df['Location'].map(location_premiums) * \
                   df['Industry'].map(industry_premiums) * \
                   df['CompanySize'].map(company_size_premiums)

    noise = np.random.normal(0, 0.08, n_samples)
    df['Salary'] *= (1 + noise)
    df['Salary'] = df['Salary'].astype(int)

    print("Synthetic data generated successfully.")
    return df

df = generate_synthetic_data(n_samples=1000)
X = df.drop('Salary', axis=1)
y = df['Salary']

# ==============================================================================
# 2. DEFINE THE PREPROCESSING AND MODEL PIPELINE
# ==============================================================================
numerical_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================================================================
# 3. HYPERPARAMETER TUNING WITH GridSearchCV
# ==============================================================================
print("\nStarting hyperparameter tuning with GridSearchCV...")
start_time = time.time()

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

end_time = time.time()
print(f"Hyperparameter tuning finished in {end_time - start_time:.2f} seconds.")

best_model = grid_search.best_estimator_
print("\nBest parameters found:", grid_search.best_params_)

# ==============================================================================
# 4. EVALUATE THE FINAL, TUNED MODEL
# ==============================================================================
print("\nEvaluating the best model on the test set...")
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Model Performance ---")
print(f"R-squared (R²): {r2:.3f}")
print(f"Mean Absolute Error (MAE): ₹{mae:,.0f}")
print(f"-> The model explains {r2:.1%} of the variance in salary.")
print(f"-> On average, the model's prediction is off by approximately ₹{mae:,.0f}.")

# ==============================================================================
# 5. SAVE THE ENHANCED MODEL
# ==============================================================================
model_filename = 'enhanced_salary_model.pkl'
joblib.dump(best_model, model_filename)
print(f"\n✅ Enhanced model successfully trained and saved as '{model_filename}'")

--- Enhanced Model Training Script ---
Generating 1000 synthetic data samples...
Synthetic data generated successfully.

Starting hyperparameter tuning with GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Hyperparameter tuning finished in 21.27 seconds.

Best parameters found: {'regressor__max_depth': 20, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}

Evaluating the best model on the test set...

--- Model Performance ---
R-squared (R²): 0.860
Mean Absolute Error (MAE): ₹196,720
-> The model explains 86.0% of the variance in salary.
-> On average, the model's prediction is off by approximately ₹196,720.

✅ Enhanced model successfully trained and saved as 'enhanced_salary_model.pkl'
